# 04 · M2: EfficientNet-B0 on log-mel, 5-fold out-of-fold

GPU + Internet on (ImageNet weights are downloaded by `timm`). **Run with Save Version → Save & Run All.**

Inputs: the aux dataset (`folds.csv`, `species.csv`) and the output of `02` (`mel.f16`, `chunks.csv`, `mel_config.json`).

Procedure (as pre-registered):
1. **Epoch selection** on fold 0 only: inside its training part, hold out 20% of recordists as validation; train up to `MAX_EPOCHS`; pick the epoch with the best recording-level macro F1.
2. **Out-of-fold**: for each fold k, train on folds ≠ k for exactly that many epochs, predict fold k.

Recording order and class order are the same as in `03` (`recordings.csv`, `classes.json`).

Outputs: `probs_m2.npy` `(n_recordings, n_classes)`, `recordings_m2.csv`, `epoch_selection.json`, `train_log.csv`.

In [ ]:
import json, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

ON_KAGGLE = Path('/kaggle/input').exists()
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT = Path('/kaggle/input') if ON_KAGGLE else REPO

def find_dir(marker):
    hits = sorted(INPUT.rglob(marker))
    assert hits, f'{marker} not found under {INPUT}'
    return hits[0].parent

AUX = find_dir('species.csv')
FEAT = find_dir('mel.f16')
OUT = Path('/kaggle/working') if ON_KAGGLE else REPO / 'cache' / 'cnn'
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

HP = dict(seed=0, max_epochs=15, batch=128, lr=1e-3, weight_decay=1e-2, drop_rate=0.2,
          freq_mask=16, time_mask=40, val_frac=0.2)
if not ON_KAGGLE:                       # quick local smoke test
    HP.update(max_epochs=2, batch=32)
print(DEVICE, AUX, FEAT, HP)

In [ ]:
species = pd.read_csv(AUX / 'species.csv', index_col=0)
folds = pd.read_csv(AUX / 'folds.csv')
chunks = pd.read_csv(FEAT / 'chunks.csv')
cfg = json.loads((FEAT / 'mel_config.json').read_text())

classes = sorted(species.index)
cls_idx = {c: i for i, c in enumerate(classes)}
rec = folds[folds['filename'].isin(chunks['filename'])].reset_index(drop=True)   # same order as 03
rec['label_idx'] = rec['primary_label'].map(cls_idx)
chunks['rec'] = chunks['filename'].map({f: i for i, f in enumerate(rec['filename'])})
chunks['y'] = rec['label_idx'].to_numpy()[chunks['rec']]

shape = (cfg['n_chunks'], cfg['n_mels'], cfg['n_frames'])
assert len(chunks) == cfg['n_chunks']
mel = np.memmap(FEAT / 'mel.f16', dtype=np.float16, mode='r', shape=shape)
try:
    import psutil
    if psutil.virtual_memory().available > mel.nbytes * 1.3:
        t0 = time.time(); mel = np.array(mel); print(f'loaded into RAM in {time.time() - t0:.0f}s')
except ImportError:
    pass
print(len(classes), 'classes |', len(rec), 'recordings |', shape, 'chunks |', round(mel.nbytes / 1e9, 1), 'GB')

## Model, batches, augmentation

In [ ]:
import timm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold

def make_model():
    return timm.create_model('efficientnet_b0', pretrained=ON_KAGGLE, in_chans=1,
                             num_classes=len(classes), drop_rate=HP['drop_rate']).to(DEVICE)

def to_input(rows):
    x = torch.from_numpy(np.asarray(mel[np.sort(rows)], dtype=np.float32)).to(DEVICE).unsqueeze(1)
    return (x - x.mean(dim=(2, 3), keepdim=True)) / (x.std(dim=(2, 3), keepdim=True) + 1e-5), np.sort(rows)

def augment(x):
    b, _, f, t = x.shape
    shift = torch.randint(0, t, (b,), device=x.device)                      # circular time shift
    idx = (torch.arange(t, device=x.device)[None, :] + shift[:, None]) % t
    x = torch.gather(x, 3, idx[:, None, None, :].expand(b, 1, f, t))
    for size, dim, param in [(f, 2, HP['freq_mask']), (t, 3, HP['time_mask'])]:   # SpecAugment, one mask each
        w = torch.randint(0, param + 1, (b,), device=x.device)
        s = (torch.rand(b, device=x.device) * (size - w)).long()
        pos = torch.arange(size, device=x.device)[None, :]
        m = (pos >= s[:, None]) & (pos < (s + w)[:, None])
        x = x.masked_fill(m[:, None, :, None] if dim == 2 else m[:, None, None, :], 0.0)
    return x

def train(train_recs, epochs, eval_recs=None, tag=''):
    torch.manual_seed(HP['seed']); rng = np.random.default_rng(HP['seed'])
    model = make_model()
    rows = chunks.index[chunks['rec'].isin(train_recs)].to_numpy()
    steps = epochs * int(np.ceil(len(rows) / HP['batch']))
    opt = torch.optim.AdamW(model.parameters(), lr=HP['lr'], weight_decay=HP['weight_decay'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=HP['lr'], total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda')
    log = []
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot = 0.0
        for b in np.array_split(rng.permutation(rows), int(np.ceil(len(rows) / HP['batch']))):
            x, srt = to_input(b)
            y = torch.from_numpy(chunks['y'].to_numpy()[srt]).to(DEVICE)
            with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
                loss = F.cross_entropy(model(augment(x)), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(b)
        entry = dict(tag=tag, epoch=ep, loss=tot / len(rows), sec=round(time.time() - t0))
        if eval_recs is not None:
            P = predict(model, eval_recs)
            entry['val_macro_f1'] = f1_score(rec.loc[eval_recs, 'label_idx'], P.argmax(1), average='macro')
        log.append(entry); print(entry)
    return model, log

@torch.no_grad()
def predict(model, recs):
    model.eval()
    rows = chunks.index[chunks['rec'].isin(recs)].to_numpy()
    out = np.zeros((len(rows), len(classes)), dtype=np.float32)
    for b in np.array_split(np.arange(len(rows)), int(np.ceil(len(rows) / (2 * HP['batch'])))):
        x, _ = to_input(rows[b])          # rows are sorted, so order is preserved
        with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
            out[b] = F.softmax(model(x).float(), dim=1).cpu().numpy()
    r = chunks.loc[rows, 'rec'].to_numpy()
    pos = {v: i for i, v in enumerate(recs)}
    agg = np.zeros((len(recs), len(classes))); cnt = np.zeros(len(recs))
    np.add.at(agg, [pos[v] for v in r], out); np.add.at(cnt, [pos[v] for v in r], 1)
    return agg / cnt[:, None]

## 1. Epoch selection (fold 0, recordist-grouped inner validation)

In [ ]:
inner = rec[rec['fold'] != 0]
sgkf = StratifiedGroupKFold(n_splits=int(round(1 / HP['val_frac'])), shuffle=True, random_state=HP['seed'])
tr_i, va_i = next(sgkf.split(inner, inner['label_idx'], groups=inner['group']))
tr_recs, va_recs = np.sort(inner.index[tr_i].to_numpy()), np.sort(inner.index[va_i].to_numpy())
assert not set(rec.loc[tr_recs, 'group']) & set(rec.loc[va_recs, 'group'])
print('inner train', len(tr_recs), '| inner val', len(va_recs))

_, sel_log = train(tr_recs, HP['max_epochs'], eval_recs=va_recs, tag='select')
best = max(sel_log, key=lambda e: e['val_macro_f1'])
EPOCHS = best['epoch']
(OUT / 'epoch_selection.json').write_text(json.dumps(dict(HP=HP, epochs=EPOCHS, log=sel_log), indent=2))
print('selected epochs:', EPOCHS, '| inner-val macro F1:', round(best['val_macro_f1'], 3))

## 2. Out-of-fold training and prediction

In [ ]:
probs = np.zeros((len(rec), len(classes)), dtype=np.float32)
logs = list(sel_log)
for k in sorted(rec['fold'].unique()):
    tr = np.sort(rec.index[rec['fold'] != k].to_numpy())
    te = np.sort(rec.index[rec['fold'] == k].to_numpy())
    model, log = train(tr, EPOCHS, tag=f'fold{k}')
    probs[te] = predict(model, te)
    logs += log
    np.save(OUT / 'probs_m2.npy', probs)            # saved after every fold
    pd.DataFrame(logs).to_csv(OUT / 'train_log.csv', index=False)
    acc = (probs[te].argmax(1) == rec.loc[te, 'label_idx']).mean()
    print(f'fold {k}: test acc {acc:.3f}')
    del model; torch.cuda.empty_cache() if DEVICE == 'cuda' else None

In [ ]:
rec[['filename', 'primary_label', 'label_idx', 'fold', 'group', 'collection', 'leak']].to_csv(OUT / 'recordings_m2.csv', index=False)
(OUT / 'classes.json').write_text(json.dumps(classes))
assert np.allclose(probs.sum(1), 1, atol=1e-3), 'some recordings were not predicted'

from sklearn.metrics import top_k_accuracy_score
y = rec['label_idx'].to_numpy()
print(f"m2  acc={np.mean(probs.argmax(1) == y):.3f}  macroF1={f1_score(y, probs.argmax(1), average='macro', labels=np.arange(len(classes))):.3f}  "
      f"top5={top_k_accuracy_score(y, probs, k=5, labels=np.arange(len(classes))):.3f}")